# PS3 Door — ExtraTrees 300

Raw Train data → timestamp-gap segmentation → event features → ExtraTrees 300. Evaluation is four chronological whole-cycle folds using the task's IoU-weighted F1.

In [1]:
from datetime import datetime
from pathlib import Path
import os
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesClassifier

def find_data():
    override = os.environ.get('PS3_ROOT')
    candidates = []
    if override:
        root = Path(override)
        candidates += [root, root / '02_Datasets' / 'Door']
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidates += [base / 'data', base / 'Door' / 'data',
                       base / 'NebulaX-Hackathon-ProblemStatement' / 'PS3' / '02_Datasets' / 'Door']
    for candidate in candidates:
        if (candidate / 'Train.csv').is_file() and (candidate / 'Train_Segments_Answer.csv').is_file():
            return candidate
    raise FileNotFoundError('Set PS3_ROOT or run this notebook from the repository')

DATA = find_data()
print('Door data:', DATA)

PS3: c:\000NebulaX\NebulaX-Hackathon-ProblemStatement\PS3


In [2]:
def ts(value):
    parts = [int(x) for x in str(value).split('-')]
    if len(parts) != 7:
        raise ValueError(f'Unexpected Door timestamp: {value!r}')
    y, mo, d, h, minute, second, millisecond = parts
    return datetime(y, mo, d, h, minute, second, millisecond * 1000).timestamp()

frame = pd.read_csv(DATA / 'Train.csv')
answers = pd.read_csv(DATA / 'Train_Segments_Answer.csv')
times = np.asarray([ts(v) for v in frame['Datetime']], dtype=float)
raw = frame.iloc[:, 1:].to_numpy(dtype=float)
assert np.all(np.diff(times) > 0) and np.isfinite(raw).all()

def iou(a0, a1, b0, b1):
    intersection = max(0.0, min(a1, b1) - max(a0, b0))
    union = max(a1, b1) - min(a0, b0)
    return intersection / union if union > 0 else 0.0

gold = [{'start': ts(r.start_time), 'end': ts(r.end_time), 'label': r.status}
        for r in answers.itertuples(index=False)]
cuts = np.flatnonzero(np.diff(times) > 0.1) + 1
bounds = np.r_[0, cuts, len(frame)]
events = [{'a': int(a), 'b': int(b - 1), 'start': float(times[a]), 'end': float(times[b - 1])}
          for a, b in zip(bounds[:-1], bounds[1:])]
assert len(events) == len(gold), (len(events), len(gold))
for event in events:
    overlaps = [iou(event['start'], event['end'], g['start'], g['end']) for g in gold]
    j = int(np.argmax(overlaps))
    if overlaps[j] <= 0.99:
        raise ValueError('Detected event does not match a labelled cycle')
    event['label'] = gold[j]['label']
print(f'raw rows={len(frame)}, detected cycles={len(events)}')

raw rows=18036, detected cycles=110


In [3]:
def make_features(events):
    rows = []
    for e in events:
        v = raw[e['a']:e['b'] + 1]
        stats = [v.mean(0), v.std(0), v.min(0), v.max(0),
                 np.quantile(v, .25, axis=0), np.median(v, axis=0),
                 np.quantile(v, .75, axis=0), np.sqrt((v ** 2).mean(0)),
                 np.abs(v).mean(0), v[0], v[-1], v[-1] - v[0],
                 np.abs(np.diff(v, axis=0)).sum(0)]
        rows.append(np.r_[np.concatenate(stats), e['end'] - e['start'], len(v)])
    return np.asarray(rows, dtype=float)

X = make_features(events)
LABELS = np.asarray(['Normal', 'Abnormal resistance'])
y = np.asarray([int(e['label'] == 'Abnormal resistance') for e in events])
truth = [{'start': e['start'], 'end': e['end'], 'label': LABELS[t]}
         for e, t in zip(events, y)]

def official_score(true_events, predicted_events, predicted_labels):
    candidates = []
    for ti, truth_event in enumerate(true_events):
        for pi, label in enumerate(predicted_labels):
            if truth_event['label'] == label:
                value = iou(truth_event['start'], truth_event['end'], predicted_events[pi]['start'], predicted_events[pi]['end'])
                if value > 0:
                    candidates.append((value, ti, pi))
    used_true, used_pred, total = set(), set(), 0.0
    for value, ti, pi in sorted(candidates, reverse=True):
        if ti not in used_true and pi not in used_pred:
            used_true.add(ti); used_pred.add(pi); total += value
    return 2.0 * total / (len(true_events) + len(predicted_labels))

def model():
    return ExtraTreesClassifier(n_estimators=300, min_samples_leaf=3,
        class_weight='balanced', max_features='sqrt', n_jobs=1, random_state=17)

blocks = np.array_split(np.arange(len(y)), 4)
oof = np.full(len(y), -1, dtype=int)
fold_train_scores = []
for fold, validation in enumerate(blocks):
    fitting = np.concatenate([b for i, b in enumerate(blocks) if i != fold])
    fitted = model().fit(X[fitting], y[fitting])
    oof[validation] = fitted.predict(X[validation])
    fold_train_scores.append(official_score([truth[i] for i in fitting], [events[i] for i in fitting], LABELS[fitted.predict(X[fitting])]))

final_model = model().fit(X, y)
train_score = official_score(truth, events, LABELS[final_model.predict(X)])
evaluation_score = official_score(truth, events, LABELS[oof])
print(f'Train IoU-weighted F1 (all-Train fit, optimistic): {train_score:.6f}')
print(f'Evaluation IoU-weighted F1 (4-fold chronological OOF): {evaluation_score:.6f}')
print(f'Fold Train scores: {[round(v, 6) for v in fold_train_scores]}')

Train IoU-weighted F1 (all-Train fit, optimistic): 1.000000
Evaluation IoU-weighted F1 (4-fold chronological OOF): 1.000000
Fold Train scores: [1.0, 1.0, 1.0, 1.0]
